## 课程三：韵律控制（45%）

实验目的：了解 VITS 模型的结构和工作原理，学习 MeloTTS 的韵律控制机制

MeloTTS 的模型架构和实现参考了 [VITS](https://github.com/jaywalnut310/vits), [VITS2](https://github.com/daniilrobnikov/vits2) and [Bert-VITS2](https://github.com/fishaudio/Bert-VITS2) 等开源项目。在本次课程中，你需要了解 VITS 模型的结构和工作原理，并尝试控制 MeloTTS 的韵律。

请阅读和观看以下资料：
- [VITS 论文](https://arxiv.org/abs/2106.06103)
- [VITS 讲解视频](https://www.bilibili.com/video/BV1Jb4y1g7q6/?p=3&share_source=copy_web&vd_source=d0dab92918e0499fec84fd963e2a879e)

![vits](./vits.png)

> 【问题一】 请介绍一下 VITS 模型的组成结构，训练和推理过程的工作原理。

**VITS 模型的组成结构：**

VITS（Variational Inference with adversarial learning for end-to-end Text-to-Speech）是一个端到端的 TTS 模型，由以下核心模块组成：

1. **文本编码器（Text Encoder）**：将输入的音素序列（phones）通过 Transformer 编码器编码为隐表示，输出均值 `m_p` 和对数方差 `logs_p`，构成先验分布 `N(m_p, exp(logs_p))`。

2. **后验编码器（Posterior Encoder）**：在训练时，将真实语音的线性谱通过 WaveNet 编码器编码为隐变量 `z`，输出均值 `m_q` 和对数方差 `logs_q`，构成后验分布 `N(m_q, exp(logs_q))`。

3. **归一化流（Normalizing Flow）**：一组可逆变换，用于将简单的先验分布映射为复杂的后验分布，缩小先验和后验之间的差距。在推理时，将从先验采样的 `z_p` 反向变换为 `z`。

4. **时长预测器（Duration Predictor）**：预测每个音素对应的帧数（时长），包含确定性时长预测器（DP）和随机时长预测器（SDP）。

5. **HiFi-GAN 解码器（Decoder）**：基于 HiFi-GAN v1 的生成器，将隐变量 `z` 解码为波形音频。

6. **多周期判别器（Multi-Period Discriminator, MPD）** 和 **多尺度判别器（Multi-Scale Discriminator, MSD）**：在训练时用于对抗学习，提升生成语音的质量。

**训练过程：**

训练采用 VAE + GAN 的混合框架：
- **VAE 损失**：包括后验编码器的 KL 散度损失（约束后验接近先验）、时长预测器的损失（使用 MAS 算法获得的对齐作为监督信号）。
- **GAN 损失**：MPD 和 MSD 对生成语音和真实语音进行判别，通过对抗损失提升语音质量。
- **重建损失**：Mel 谱重建损失，确保生成语音的频谱特征接近真实语音。
- 各模块交替优化，共同训练。

**推理过程：**

1. 输入音素序列经过文本编码器得到先验分布参数 `m_p, logs_p`。
2. 时长预测器（DP + SDP 混合）预测每个音素的时长 `w_ceil`。
3. 根据 `w_ceil` 将先验分布参数展开到帧级别（通过注意力矩阵 `attn`）。
4. 从先验分布采样 `z_p = m_p + randn * exp(logs_p) * noise_scale`。
5. 通过归一化流反向变换得到 `z = flow(z_p)`。
6. HiFi-GAN 解码器将 `z` 解码为波形音频。

> 【问题二】 请介绍一下 VITS 中的 Monotonic Alignment Search (MAS) 算法和 Stochastic Duration Predictor 模块，说明它们在时长建模和韵律生成中的作用。

**Monotonic Alignment Search (MAS) 算法：**

MAS 是 VITS 中用于在训练阶段自动获取文本和语音之间单调对齐关系的算法。其核心思想是：在语音合成中，音素序列和语音帧之间满足**单调、连续**的对齐约束（即文本从左到右对应语音从左到右，不出现交叉或跳跃）。

**算法原理：**
- 给定文本编码器输出的先验分布 `N(m_p, logs_p)` 和后验编码器输出的隐变量 `z`，MAS 通过动态规划在对数域中搜索使后验概率最大的单调对齐路径。
- 具体地，对于每一帧 `t`，从上一帧的对齐位置开始（单调性约束），计算所有可能对齐位置的对数概率，选取最大值对应的路径。
- 时间复杂度为 `O(T * S)`，其中 `T` 是语音帧数、`S` 是音素数。

**作用：**
- MAS 在训练中提供文本-语音的硬对齐（hard alignment），作为时长预测器的监督信号。每个音素对应的帧数（时长）由 MAS 的对齐结果直接确定。
- 这些时长标签用于训练确定性时长预测器（DP），使模型学会预测每个音素的持续时间。

**Stochastic Duration Predictor (SDP) 模块：**

SDP 是一个基于归一化流的随机时长预测器，用于建模音素时长的概率分布。

**工作原理：**
- SDP 以文本编码器的隐表示和说话人嵌入为条件，通过归一化流将简单的高斯分布映射为复杂的时长分布。
- 在训练时，使用 MAS 得到的真实时长作为目标，通过最大似然训练 SDP。
- 在推理时，从标准正态分布采样，经流变换得到时长预测值。通过 `noise_scale_w` 参数控制采样噪声，调节时长的多样性。

**在韵律生成中的作用：**
- DP 提供确定性的时长预测，保证合成的稳定性；SDP 提供随机性的时长预测，增加韵律的多样性和自然度。
- 两者通过 `sdp_ratio` 参数混合：`logw = sdp(...) * sdp_ratio + dp(...) * (1 - sdp_ratio)`。
- SDP 使得模型在多次合成同一句话时可以产生不同的韵律模式，更接近真人说话的随机性。结合 `w_ceil` 的调整，可以实现精细的韵律控制（如重音、停顿、语速变化等）。

MeloTTS 使用 Stochastic Duration Predictor 模块来实现韵律控制机制的代码在 `melo/models.py` 的 `infer` 函数中（966 行），其中一个重要的输出是 `w_ceil`，它表示了每个音素的持续时间。可以通过自定义的 `get_original_w_ceil` 函数来获取 `w_ceil`：

In [1]:
from melo.api import TTS
import pandas as pd

pd.set_option('display.max_columns', None)

# Speed is adjustable
speed = 1
device = 'cuda:0' # or cuda:0

text = "落霞与孤鹜齐飞，秋水共长天一色。"
model = TTS(language='ZH', device=device)
speaker_ids = model.hps.data.spk2id

output_path = 'zh.wav'

# 获取原始的 phones、tones、w_ceil 列表
w_ceil_list, phone_list, tone_list = model.get_original_w_ceil(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0)

# 将 phones 列表中的 ID 转换为对应的符号
symbol_to_id_map = model.symbol_to_id
id_to_symbol_map = {v: k for k, v in zip(symbol_to_id_map.keys(), symbol_to_id_map.values())}
print(f'id_to_symbol_map: {id_to_symbol_map}')
# for w_ceil, phones, tones in zip(w_ceil_list, phone_list, tone_list):
#     print('phones:', phones.flatten().tolist())
#     print('tones:', tones.flatten().tolist())
#     print('w_ceil:', w_ceil.flatten().int().tolist())

print(f'w_ceil_list: {w_ceil_list}')

df = pd.DataFrame({
    'phones': [id_to_symbol_map.get(item, '') for sublist in phone_list for item in sublist.flatten().tolist()],
    'tones': [item for sublist in tone_list for item in sublist.flatten().tolist()],
    'w_ceil': [item for sublist in w_ceil_list for item in sublist.flatten().int().tolist()]
})


df.T

C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\google\api_core\_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.23). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\google\auth\__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\google\oauth2\__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth

C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


 > Text split to sentences.
落霞与孤鹜齐飞, 秋水共长天一色.
 > ===========================


  0%|          | 0/1 [00:00<?, ?it/s]

...

Some weights of the model checkpoint at bert-base-multilingual-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


100%|██████████| 1/1 [00:13<00:00, 13.68s/it]

100%|██████████| 1/1 [00:13<00:00, 13.68s/it]

id_to_symbol_map: {0: '_', 1: 'AA', 2: 'E', 3: 'EE', 4: 'En', 5: 'N', 6: 'OO', 7: 'V', 8: 'a', 9: 'a:', 10: 'aa', 11: 'ae', 12: 'ah', 13: 'ai', 14: 'an', 15: 'ang', 16: 'ao', 17: 'aw', 18: 'ay', 19: 'b', 20: 'by', 21: 'c', 22: 'ch', 23: 'd', 24: 'dh', 25: 'dy', 26: 'e', 27: 'e:', 28: 'eh', 29: 'ei', 30: 'en', 31: 'eng', 32: 'er', 33: 'ey', 34: 'f', 35: 'g', 36: 'gy', 37: 'h', 38: 'hh', 39: 'hy', 40: 'i', 41: 'i0', 42: 'i:', 43: 'ia', 44: 'ian', 45: 'iang', 46: 'iao', 47: 'ie', 48: 'ih', 49: 'in', 50: 'ing', 51: 'iong', 52: 'ir', 53: 'iu', 54: 'iy', 55: 'j', 56: 'jh', 57: 'k', 58: 'ky', 59: 'l', 60: 'm', 61: 'my', 62: 'n', 63: 'ng', 64: 'ny', 65: 'o', 66: 'o:', 67: 'ong', 68: 'ou', 69: 'ow', 70: 'oy', 71: 'p', 72: 'py', 73: 'q', 74: 'r', 75: 'ry', 76: 's', 77: 'sh', 78: 't', 79: 'th', 80: 'ts', 81: 'ty', 82: 'u', 83: 'u:', 84: 'ua', 85: 'uai', 86: 'uan', 87: 'uang', 88: 'uh', 89: 'ui', 90: 'un', 91: 'uo', 92: 'uw', 93: 'v', 94: 'van', 95: 've', 96: 'vn', 97: 'w', 98: 'x', 99: 'y', 100: 

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64
phones,_,_,_,l,_,uo,_,x,_,ia,_,y,_,v,_,g,_,u,_,w,_,u,_,q,_,i,_,f,_,ei,_,",",_,q,_,iu,_,sh,_,ui,_,g,_,ong,_,ch,_,ang,_,t,_,ian,_,y,_,i,_,s,_,e,_,.,_,_,_
tones,0,0,0,4,0,4,0,2,0,2,0,3,0,3,0,1,0,1,0,4,0,4,0,2,0,2,0,1,0,1,0,0,0,1,0,1,0,3,0,3,0,4,0,4,0,2,0,2,0,1,0,1,0,2,0,2,0,4,0,4,0,0,0,0,0
w_ceil,7,7,15,2,3,3,2,6,4,5,3,4,4,3,6,3,3,3,3,3,3,3,3,11,4,3,3,4,6,6,4,4,7,5,2,5,3,8,4,4,4,3,4,4,3,4,4,6,4,5,3,3,3,3,4,3,5,6,5,5,4,3,3,3,4


从以上 pandas DataFrame 中可以看到每个音素对应的持续时间（w_ceil），w_ceil 的一个单位对应 512 (hop_length) / 44100 (采样率) ≈ 0.0116s = 11.6ms 的时间长度。因此，可以通过调整 `w_ceil_list` 中每个 w_ceil 的值，来控制每个音素的持续时间，从而实现简单的韵律控制。

In [2]:
# w_ceil_list = [[7, 7, 15, 2, 3, 3, 2, 6, 4, 5, 3, 4, 4, 3, 6, 3, 9, 9, 9, 9, 9, 9, 9, 11, 4, 3, 3, 4, 6, 6, 4, 4, 7, 5, 2, 5, 3, 8, 4, 4, 4, 3, 4, 4, 3, 8, 8, 18, 12, 15, 12, 12, 12, 3, 4, 3, 5, 6, 5, 5, 4, 3, 3, 3, 4]]
# w_ceil_list = [[7, 7, 15, 2, 3, 3, 2, 6, 4, 5, 3, 4, 4, 3, 6, 3, 9, 9, 9, 9, 9, 9, 9, 11, 4, 3, 3, 4, 6, 6, 4, 4, 7, 5, 2, 5, 3, 8, 4, 4, 4, 3, 4, 4, 3, 8, 8, 18, 12, 15, 12, 12, 12, 3, 4, 3, 5, 6, 5, 5, 4, 3, 3, 3, 4]]
modified_w_ceil_list = w_ceil_list[0].squeeze(0).int().tolist()
modified_w_ceil_list[0][15:23] = [9] * 8 # 延长 “孤” 和 “鹜” 音素的持续时间为 9 个单位（约 104ms）
modified_w_ceil_list[0][45:53] = [9] * 8 # 延长 “长” 和 “天” 音素的持续时间为 9 个单位（约 104ms）

# 修改后的 w_ceil_list
print(f'modified w_ceil_list: {modified_w_ceil_list}')


modified w_ceil_list: [[7, 7, 15, 2, 3, 3, 2, 6, 4, 5, 3, 4, 4, 3, 6, 9, 9, 9, 9, 9, 9, 9, 9, 11, 4, 3, 3, 4, 6, 6, 4, 4, 7, 5, 2, 5, 3, 8, 4, 4, 4, 3, 4, 4, 3, 9, 9, 9, 9, 9, 9, 9, 9, 3, 4, 3, 5, 6, 5, 5, 4, 3, 3, 3, 4]]


In [3]:
# 按照原始的韵律生成语音，关闭随机性以便对比
model.tts_to_file(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0)

from IPython.display import Audio

Audio(output_path)

 > Text split to sentences.
落霞与孤鹜齐飞, 秋水共长天一色.
 > ===========================


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  2.58it/s]

100%|██████████| 1/1 [00:00<00:00,  2.57it/s]

In [4]:
# 按照调整后的韵律生成语音，使用自定义的 w_ceil_list 来控制每个音素的持续时间，关闭随机性以便对比
model.tts_to_file_custom_duration(text, speaker_ids['ZH'], output_path, speed=speed, sdp_ratio=0, noise_scale=0, noise_scale_w=0, w_ceil_customized=modified_w_ceil_list)

from IPython.display import Audio

Audio(output_path)

 > Text split to sentences.
落霞与孤鹜齐飞, 秋水共长天一色.
 > ===========================


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  6.21it/s]

100%|██████████| 1/1 [00:00<00:00,  6.17it/s]

可以观察到修改后的音频中，"孤"和"鹜"这两个字的音素的持续时间被延长了，"长"和"天"这两个字的音素的持续时间也被延长了，从而产生更突出的节奏和停连效果。

> 【问题三】 请通过调整 `w_ceil` 来控制合成语音的韵律，合成一句 “你听到了吗？这句话我会说得越来越慢。” 的语音。要求语速越来越慢：句首接近原始时长、句尾约为原始时长的 3 倍（对应语速约从 1 倍`线性`降至 0.33 倍）。提交代码和生成的音频文件（命名为 `homework_3.wav`）。

In [5]:
from melo.api import TTS
import numpy as np

speed = 1
device = 'cuda:0'

text = "你听到了吗？这句话我会说得越来越慢。"
model = TTS(language='ZH', device=device)
speaker_ids = model.hps.data.spk2id

output_path = 'homework_3.wav'

# 获取原始的 w_ceil 列表
w_ceil_list, phone_list, tone_list = model.get_original_w_ceil(
    text, speaker_ids['ZH'], output_path, speed=speed,
    sdp_ratio=0, noise_scale=0, noise_scale_w=0
)

# 将 w_ceil 转为可修改的列表
modified_w_ceil = w_ceil_list[0].squeeze(0).int().tolist()

# 计算总音素数，线性插值使语速从 1 倍逐渐降至约 0.33 倍（即时长从 1x 线性增至 3x）
n = len(modified_w_ceil[0])
for i in range(n):
    scale = 1.0 + 2.0 * i / (n - 1)  # 线性从 1.0 到 3.0
    modified_w_ceil[0][i] = max(1, int(round(modified_w_ceil[0][i] * scale)))

print(f'原始 w_ceil: {w_ceil_list[0].squeeze(0).int().tolist()}')
print(f'修改后 w_ceil: {modified_w_ceil}')

# 使用自定义 w_ceil 合成语音
model.tts_to_file_custom_duration(
    text, speaker_ids['ZH'], output_path, speed=speed,
    sdp_ratio=0, noise_scale=0, noise_scale_w=0,
    w_ceil_customized=modified_w_ceil
)

C:\Users\Lenovo\miniconda3\envs\poetry\lib\site-packages\torch\nn\utils\weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


 > Text split to sentences.
你听到了吗. 这句话我会说得越来越慢.
 > ===========================


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 10.33it/s]

原始 w_ceil: [[7, 10, 13, 2, 3, 3, 3, 7, 3, 2, 4, 3, 3, 3, 3, 2, 2, 2, 5, 5, 5, 3, 5, 3, 8, 4, 3, 2, 2, 3, 3, 3, 2, 4, 3, 4, 3, 2, 3, 2, 2, 3, 4, 3, 4, 4, 3, 3, 2, 2, 3, 3, 5, 3, 6, 5, 3, 3, 3, 3, 3, 3, 3, 3, 7, 3, 5, 4, 4, 3, 3, 3, 4]]
修改后 w_ceil: [[7, 10, 14, 2, 3, 3, 4, 8, 4, 2, 5, 4, 4, 4, 4, 3, 3, 3, 8, 8, 8, 5, 8, 5, 13, 7, 5, 4, 4, 5, 6, 6, 4, 8, 6, 8, 6, 4, 6, 4, 4, 6, 9, 7, 9, 9, 7, 7, 5, 5, 7, 7, 12, 7, 15, 13, 8, 8, 8, 8, 8, 8, 8, 8, 19, 8, 14, 11, 12, 9, 9, 9, 12]]
 > Text split to sentences.
你听到了吗. 这句话我会说得越来越慢.
 > ===========================


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  3.08it/s]

100%|██████████| 1/1 [00:00<00:00,  3.07it/s]